# Semaine 1 — Requêtes SQL sur une base d'assurance maladie fictive

**Auteur :** Abel Bebe Koum — Gestionnaire Maîtrise des Risques, en transition vers l'analyse de données.

Ce notebook rejoue mes 15 requêtes de la semaine 1 sur la base `cpam_training_v1.db` (SQLite, données 100 % fictives) et affiche leur résultat. Thèmes : agrégats, jointures, contrôles de cohérence des indemnités journalières, doublons, délai de carence, synthèse des anomalies non détectées et classement des prescripteurs.

Les fichiers `.sql` du même dossier contiennent chaque requête seule.

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

# Cherche la base dans le dépôt, quel que soit l'endroit d'où le notebook est lancé
candidats = [Path('../../data/v1/cpam_training_v1.db'), Path('../../cpam_training_v1.db'), Path('cpam_training_v1.db')]
DB = next(p for p in candidats if p.exists())
con = sqlite3.connect(DB)
pd.set_option('display.max_rows', 20)

def requete(sql):
    """Exécute une requête SQL et renvoie le résultat sous forme de tableau."""
    return pd.read_sql(sql, con)

print(f'Base chargée : {DB}')

Base chargée : ../../cpam_training_v1.db


## J1 · Agrégats simples

### Question 1

Nombre d'assurés par département et par sexe, trié par département puis sexe.

In [2]:
requete("""
SELECT
    departement,
    sexe,
    COUNT(*) AS nombre_assures
FROM assures
GROUP BY departement, sexe
ORDER BY departement ASC, sexe ASC
""")

,departement,sexe,nombre_assures
0,38,F,22
1,38,M,27
2,69,F,4
3,69,M,7


### Question 2

Arrêts reçus hors délai (plus de 48 h, soit plus de 2 jours) : `id_arret`, `id_assure`, `date_debut`, `date_reception`, nombre de jours de retard. Tri par retard décroissant.

In [3]:
requete("""
SELECT
    id_arret,
    id_assure,
    date_debut,
    date_reception,
    CAST(julianday(date_reception) - julianday(date_debut) AS INTEGER) AS nombre_jours_retard
FROM arrets_travail
WHERE julianday(date_reception) - julianday(date_debut) > 2
ORDER BY nombre_jours_retard DESC
""")

,id_arret,id_assure,date_debut,date_reception,nombre_jours_retard
0,AR0019,A0014,2025-08-06,2025-08-12,6
1,AR0049,A0042,2025-03-11,2025-03-17,6
2,AR0066,A0018,2025-11-13,2025-11-18,5
3,AR0071,A0005,2025-05-23,2025-05-28,5
4,AR0074,A0001,2025-03-13,2025-03-18,5
5,AR0076,A0003,2025-10-04,2025-10-09,5
6,AR0077,A0018,2025-10-21,2025-10-26,5
7,AR0079,A0036,2025-07-11,2025-07-16,5
8,AR0006,A0052,2025-10-02,2025-10-06,4
9,AR0017,A0003,2025-09-29,2025-10-03,4


### Question 3

Montant total des IJ au statut **Payé**, par type de risque, avec le nombre de paiements et le montant moyen (arrondi à 2 décimales).

In [4]:
requete("""
SELECT
    a.type_risque,
    COUNT(*) AS nombre_paiements,
    ROUND(SUM(p.montant_brut),2) AS montant_total_ij,
    ROUND(AVG(p.montant_brut), 2) AS montant_moyen
FROM paiements_ij p
INNER JOIN arrets_travail a
    ON p.id_arret = a.id_arret
WHERE p.statut = 'Payé'
GROUP BY a.type_risque
ORDER BY montant_total_ij DESC
""")

,type_risque,nombre_paiements,montant_total_ij,montant_moyen
0,Maladie,81,29048.11,358.62
1,AT,19,7999.87,421.05
2,MP,9,3992.56,443.62


## J2 · Jointures

### Question 4

Pour **chaque** employeur (y compris ceux sans arrêt) : nombre de salariés ayant eu au moins un arrêt, et nombre total de jours d'arrêt (`date_fin - date_debut + 1`).

In [5]:
requete("""
SELECT
    e.id_employeur,
    e.raison_sociale,
    COUNT(DISTINCT ar.id_assure) AS nombre_salaries_avec_arret,
    COALESCE(
        SUM(
            CAST(
                julianday(ar.date_fin) - julianday(ar.date_debut) + 1
                AS INTEGER
            )
        ),
        0
    ) AS nombre_total_jours
FROM employeurs e
LEFT JOIN assures a
    ON e.id_employeur = a.id_employeur
LEFT JOIN arrets_travail ar
    ON a.id_assure = ar.id_assure
GROUP BY
    e.id_employeur,
    e.raison_sociale
ORDER BY e.id_employeur ASC
""")

,id_employeur,raison_sociale,nombre_salaries_avec_arret,nombre_total_jours
0,E001,Entreprise A01,4,71
1,E002,Entreprise B02,2,63
2,E003,Entreprise C03,3,148
3,E004,Entreprise D04,2,84
4,E005,Entreprise E05,0,0
5,E006,Entreprise F06,1,79
6,E007,Entreprise G07,3,152
7,E008,Entreprise H08,3,159
8,E009,Entreprise I09,4,157
9,E010,Entreprise J10,1,59


### Question 5

Assurés sans médecin traitant ayant eu au moins une prestation : identifiant, commune, nombre de prestations, total remboursé.

In [6]:
requete("""
SELECT
    a.id_assure,
    a.commune,
    COUNT(pr.id_prestation) AS nombre_prestations,
    SUM(pr.montant_rembourse) AS total_rembourse
FROM assures a
INNER JOIN prestations pr
    ON a.id_assure = pr.id_assure
WHERE a.id_medecin_traitant IS NULL
GROUP BY
    a.id_assure,
    a.commune
ORDER BY total_rembourse DESC
""")

,id_assure,commune,nombre_prestations,total_rembourse
0,A0027,Pont-Évêque,2,35.18
1,A0033,Bourgoin-Jallieu,2,29.06


### Question 6

Prestations facturées par un professionnel au statut **Cessation** : détail de la prestation + spécialité du professionnel.

In [7]:
requete("""
SELECT
    p.id_prestation,
    p.id_assure,
    p.id_ps,
    ps.specialite,
    p.code_acte,
    p.date_soins,
    p.date_liquidation,
    p.montant_depense,
    p.base_remboursement,
    p.taux_applique,
    p.montant_rembourse
FROM prestations p
INNER JOIN professionnels_sante ps
    ON p.id_ps = ps.id_ps
WHERE ps.statut = 'Cessation'
ORDER BY p.date_soins ASC
""")

,id_prestation,id_assure,id_ps,specialite,code_acte,date_soins,date_liquidation,montant_depense,base_remboursement,taux_applique,montant_rembourse
0,PR00010,A0055,PS024,Laboratoire,B,2025-02-17,2025-03-04,22.5,22.5,60,13.5
1,PR00055,A0058,PS024,Laboratoire,B,2025-07-13,2025-07-16,22.5,22.5,60,13.5
2,PR00056,A0015,PS024,Laboratoire,B,2025-07-23,2025-08-22,22.5,22.5,60,13.5
3,PR00013,A0028,PS024,Laboratoire,B,2025-08-04,2025-08-05,22.5,22.5,60,13.5
4,PR00039,A0057,PS024,Laboratoire,B,2025-08-15,2025-08-16,22.5,22.5,60,13.5
5,PR00020,A0046,PS024,Laboratoire,B,2025-10-10,2025-10-12,22.5,22.5,60,13.5


## J3 · Contrôles de cohérence IJ

### Question 7

Paiements dont `montant_brut` ≠ `nb_jours × montant_journalier` (tolérance 0,01 €) : afficher l'écart.

In [8]:
requete("""
SELECT
    id_paiement,
    id_arret,
    nb_jours,
    montant_journalier,
    montant_brut,
    ABS(montant_brut - nb_jours * montant_journalier) AS ecart_paiement
FROM paiements_ij
WHERE ABS(montant_brut - nb_jours * montant_journalier) > 0.01
ORDER BY ecart_paiement DESC
""")

,id_paiement,id_arret,nb_jours,montant_journalier,montant_brut,ecart_paiement
0,P00045,AR0031,14,40.64,654.46,85.5
1,P00061,AR0042,1,59.38,101.38,42.0


### Question 8

Paiements dont la période indemnisée sort des bornes de l'arrêt (avant `date_debut` ou après `date_fin`) : nombre de jours payés à tort.

In [9]:
requete("""
SELECT
    p.id_paiement,
    a.id_assure,
    p.id_arret,
    a.date_debut,
    a.date_fin,
    p.periode_debut,
    p.periode_fin,
    p.nb_jours,
    p.nb_jours -
    MAX(0,
        julianday(MIN(p.periode_fin, a.date_fin))
        - julianday(MAX(p.periode_debut, a.date_debut))
        + 1
    ) AS jours_payes_a_tort
FROM paiements_ij p
INNER JOIN arrets_travail a
    ON p.id_arret = a.id_arret
WHERE p.periode_debut < a.date_debut
   OR p.periode_fin > a.date_fin
ORDER BY jours_payes_a_tort DESC
""")

,id_paiement,id_assure,id_arret,date_debut,date_fin,periode_debut,periode_fin,nb_jours,jours_payes_a_tort
0,P00059,A0001,AR0041,2025-03-09,2025-03-12,2025-03-12,2025-03-17,6,5.0
1,P00088,A0022,AR0065,2025-04-21,2025-04-27,2025-04-21,2025-05-02,12,5.0


### Question 9

Paiements d'arrêts **Maladie** dont le montant journalier dépasse le plafond de 41,95 € : montant versé en trop.

In [10]:
requete("""
SELECT
    p.id_paiement,
    p.id_arret,
    a.type_risque,
    p.montant_journalier,
    p.nb_jours,
    (p.montant_journalier - 41.95) * p.nb_jours AS trop_percu
FROM paiements_ij p
INNER JOIN arrets_travail a
    ON p.id_arret = a.id_arret
WHERE a.type_risque = 'Maladie'
  AND p.montant_journalier > 41.95
ORDER BY trop_percu DESC
""")

,id_paiement,id_arret,type_risque,montant_journalier,nb_jours,trop_percu
0,P00025,AR0020,Maladie,51.77,21,206.22
1,P00002,AR0002,Maladie,44.31,11,25.96


## J4 · Doublons et carence

### Question 10

Doublons de paiement : même arrêt, même période, au moins deux paiements non annulés.

In [11]:
requete("""
SELECT
    p.id_arret,
    a.id_assure,
    p.periode_debut,
    p.periode_fin,
    COUNT(*) AS nombre_paiements
FROM paiements_ij p
INNER JOIN arrets_travail a
    ON p.id_arret = a.id_arret
WHERE p.statut <> 'Annulé'
GROUP BY
    p.id_arret,
    a.id_assure,
    p.periode_debut,
    p.periode_fin
HAVING COUNT(*) >= 2
ORDER BY nombre_paiements DESC
""")

,id_arret,id_assure,periode_debut,periode_fin,nombre_paiements
0,AR0023,A0045,2025-09-26,2025-10-09,2
1,AR0038,A0047,2025-04-14,2025-04-26,2
2,AR0061,A0043,2025-03-14,2025-03-17,2


### Question 11

Paiements en période de carence : arrêt **Maladie** + **Initial** dont la période payée commence avant `date_debut + 3 jours`. Calculer l'indu.

In [12]:
requete("""
SELECT
    p.id_paiement,
    p.id_arret,
    a.id_assure,
    a.date_debut,
    p.periode_debut,
    date(a.date_debut, '+3 days') AS premier_jour_indemnisable,
    CAST(julianday(date(a.date_debut, '+3 days'))
         - julianday(p.periode_debut) AS INTEGER) AS jours_carence_payes,
    ROUND((julianday(date(a.date_debut, '+3 days'))
           - julianday(p.periode_debut)) * p.montant_journalier, 2) AS indu
FROM paiements_ij p
INNER JOIN arrets_travail a
    ON p.id_arret = a.id_arret
WHERE a.type_risque = 'Maladie'
  AND a.nature = 'Initial'
  AND p.statut <> 'Annulé'
  AND p.periode_debut < date(a.date_debut, '+3 days')
ORDER BY indu DESC
""")

,id_paiement,id_arret,id_assure,date_debut,periode_debut,premier_jour_indemnisable,jours_carence_payes,indu
0,P00025,AR0020,A0005,2025-04-02,2025-04-02,2025-04-05,3,155.31
1,P00051,AR0035,A0028,2025-07-18,2025-07-18,2025-07-21,3,125.85
2,P00053,AR0037,A0003,2025-04-05,2025-04-05,2025-04-08,3,125.85


### Question 12

Doublons de facturation : même assuré, même professionnel, même acte, même date de soins.

In [13]:
requete("""
SELECT
    p.id_assure,
    p.id_ps,
    p.code_acte,
    p.date_soins,
    COUNT(*)                                       AS nombre_facturations,
    GROUP_CONCAT(p.id_prestation, ', ')            AS prestations_concernees,
    ROUND(SUM(p.montant_rembourse), 2)             AS total_rembourse,
    ROUND(SUM(p.montant_rembourse)
          - MIN(p.montant_rembourse), 2)           AS montant_en_trop
FROM prestations p
GROUP BY p.id_assure, p.id_ps, p.code_acte, p.date_soins
HAVING COUNT(*) >= 2
ORDER BY montant_en_trop DESC
""")

,id_assure,id_ps,code_acte,date_soins,nombre_facturations,prestations_concernees,total_rembourse,montant_en_trop
0,A0019,PS025,TRA,2025-06-14,2,"PR00077, PR00114",97.5,48.75
1,A0032,PS013,CS,2025-05-30,2,"PR00097, PR00113",44.1,22.05
2,A0028,PS007,TCG,2025-02-07,2,"PR00080, PR00112",42.0,21.00


## J5 · Synthèse maîtrise des risques

### Question 13

Arrêts qui se chevauchent pour un même assuré (auto-jointure) : les deux identifiants et le nombre de jours en commun.

In [14]:
requete("""
SELECT
    a1.id_assure,
    a1.id_arret AS id_arret_1,
    a2.id_arret AS id_arret_2,
    a1.date_debut AS debut_arret_1,
    a1.date_fin AS fin_arret_1,
    a2.date_debut AS debut_arret_2,
    a2.date_fin AS fin_arret_2,

    CAST(
        julianday(
            MIN(a1.date_fin, a2.date_fin)
        )
        -
        julianday(
            MAX(a1.date_debut, a2.date_debut)
        )
        + 1 AS INTEGER
    ) AS nombre_jours_communs

FROM arrets_travail a1

INNER JOIN arrets_travail a2
    ON a1.id_assure = a2.id_assure

WHERE a1.id_arret < a2.id_arret
  AND a1.date_debut <= a2.date_fin
  AND a2.date_debut <= a1.date_fin

ORDER BY nombre_jours_communs DESC
""")

,id_assure,id_arret_1,id_arret_2,debut_arret_1,fin_arret_1,debut_arret_2,fin_arret_2,nombre_jours_communs
0,A0052,AR0006,AR0026,2025-10-02,2025-10-22,2025-09-27,2025-10-26,21
1,A0028,AR0043,AR0078,2025-08-04,2025-08-18,2025-07-25,2025-08-14,11
2,A0025,AR0015,AR0082,2025-01-23,2025-02-12,2025-01-25,2025-02-03,10
3,A0018,AR0031,AR0077,2025-10-23,2025-11-12,2025-10-21,2025-10-30,8
4,A0034,AR0013,AR0072,2025-09-13,2025-09-19,2025-09-09,2025-10-08,7
5,A0003,AR0076,AR0081,2025-10-04,2025-11-02,2025-10-01,2025-10-10,7
6,A0001,AR0025,AR0058,2025-01-28,2025-02-11,2025-02-08,2025-02-17,4
7,A0003,AR0017,AR0081,2025-09-29,2025-10-03,2025-10-01,2025-10-10,3


### Question 14

**Requête de synthèse** (CTE) : regrouper les anomalies IJ des questions 7 à 11 dans une seule liste (`id_paiement`, `type_anomalie`, `montant_indu`), puis identifier celles **absentes** de la table `controles`. Total de l'indu non détecté.

In [15]:
requete("""
WITH anomalies_ij AS (

    -- QUESTION 7 : incoherence du montant brut
    SELECT
        p.id_paiement,
        'Montant brut incoherent' AS type_anomalie,
        ABS(p.montant_brut -
            p.nb_jours * p.montant_journalier) AS montant_indu
    FROM paiements_ij p
    WHERE ABS(p.montant_brut -
              p.nb_jours * p.montant_journalier) > 0.01

    UNION ALL

    -- QUESTION 8 : paiement hors periode de l'arret
    SELECT
        p.id_paiement,
        'Paiement hors periode' AS type_anomalie,
        (
            p.nb_jours -
            MAX(0,
                julianday(MIN(p.periode_fin, a.date_fin))
                -
                julianday(MAX(p.periode_debut, a.date_debut))
                + 1
            )
        ) * p.montant_journalier AS montant_indu
    FROM paiements_ij p
    INNER JOIN arrets_travail a
        ON p.id_arret = a.id_arret
    WHERE p.periode_debut < a.date_debut
       OR p.periode_fin > a.date_fin

    UNION ALL

    -- QUESTION 9 : depassement du plafond Maladie
    SELECT
        p.id_paiement,
        'Depassement plafond IJ' AS type_anomalie,
        (p.montant_journalier - 41.95) * p.nb_jours AS montant_indu
    FROM paiements_ij p
    INNER JOIN arrets_travail a
        ON p.id_arret = a.id_arret
    WHERE a.type_risque = 'Maladie'
      AND p.montant_journalier > 41.95

    UNION ALL

    -- QUESTION 10 : doublons de paiement
    SELECT
        d.id_paiement,
        'Doublon de paiement' AS type_anomalie,
        d.montant_brut AS montant_indu
    FROM (
        SELECT
            p.*,
            ROW_NUMBER() OVER (
                PARTITION BY
                    p.id_arret,
                    p.periode_debut,
                    p.periode_fin
                ORDER BY p.id_paiement
            ) AS rang_doublon,
            COUNT(*) OVER (
                PARTITION BY
                    p.id_arret,
                    p.periode_debut,
                    p.periode_fin
            ) AS nombre_paiements
        FROM paiements_ij p
        WHERE p.statut <> 'Annulé'
    ) d
    WHERE d.nombre_paiements >= 2
      AND d.rang_doublon > 1

    UNION ALL

    -- QUESTION 11 : paiement pendant la carence
    SELECT
        p.id_paiement,
        'Paiement en periode de carence' AS type_anomalie,
        (
            julianday(
                MIN(p.periode_fin, date(a.date_debut, '+2 days'))
            )
            -
            julianday(p.periode_debut)
            + 1
        ) * p.montant_journalier AS montant_indu
    FROM paiements_ij p
    INNER JOIN arrets_travail a
        ON p.id_arret = a.id_arret
    WHERE a.type_risque = 'Maladie'
      AND a.nature = 'Initial'
      AND p.periode_debut <= date(a.date_debut, '+2 days')
      AND p.periode_fin >= a.date_debut

),

anomalies_non_detectees AS (

    SELECT
        an.id_paiement,
        an.type_anomalie,
        an.montant_indu
    FROM anomalies_ij an
    LEFT JOIN controles c
        ON c.table_source = 'paiements_ij'
       AND c.id_reference = an.id_paiement
    WHERE c.id_reference IS NULL

)

SELECT
    id_paiement,
    type_anomalie,
    ROUND(montant_indu, 2) AS montant_indu,
    ROUND(SUM(montant_indu) OVER (), 2)
        AS total_indu_non_detecte
FROM anomalies_non_detectees
ORDER BY id_paiement, type_anomalie
""")

,id_paiement,type_anomalie,montant_indu,total_indu_non_detecte
0,P00002,Depassement plafond IJ,25.96,1246.41
1,P00045,Montant brut incoherent,85.50,1246.41
2,P00059,Paiement hors periode,209.75,1246.41
3,P00061,Montant brut incoherent,42.00,1246.41
4,P00119,Doublon de paiement,760.76,1246.41
5,P00120,Doublon de paiement,122.44,1246.41


### Question 15

Classement des prescripteurs par nombre total de jours d'arrêt prescrits (`RANK()`), avec leur part en % du total.

In [16]:
requete("""
WITH jours_par_prescripteur AS (
    SELECT
        a.id_ps_prescripteur,
        ps.specialite,
        SUM(
            CAST(
                julianday(a.date_fin) -
                julianday(a.date_debut) + 1
                AS INTEGER
            )
        ) AS total_jours_arret
    FROM arrets_travail a
    INNER JOIN professionnels_sante ps
        ON a.id_ps_prescripteur = ps.id_ps
    GROUP BY
        a.id_ps_prescripteur,
        ps.specialite
)

SELECT
    id_ps_prescripteur,
    specialite,
    total_jours_arret,

    RANK() OVER (
        ORDER BY total_jours_arret DESC
    ) AS classement,

    ROUND(
        total_jours_arret * 100.0 /
        SUM(total_jours_arret) OVER (),
        2
    ) AS part_pourcentage

FROM jours_par_prescripteur
ORDER BY classement ASC
""")

,id_ps_prescripteur,specialite,total_jours_arret,classement,part_pourcentage
0,PS003,Médecin généraliste,270,1,22.65
1,PS007,Médecin généraliste,184,2,15.44
2,PS008,Médecin généraliste,146,3,12.25
3,PS002,Médecin généraliste,125,4,10.49
4,PS004,Médecin généraliste,120,5,10.07
5,PS009,Médecin généraliste,107,6,8.98
6,PS001,Médecin généraliste,97,7,8.14
7,PS005,Médecin généraliste,82,8,6.88
8,PS011,Psychiatre,38,9,3.19
9,PS012,Rhumatologue,14,10,1.17


## Bilan

- 15 requêtes exécutées sans erreur sur la base v1.
- La question 14 regroupe 5 contrôles IJ dans une CTE et isole les anomalies absentes de la table `controles` : **6 anomalies non détectées, pour 1 246,41 € d'indu**.
- La question 15 classe les prescripteurs avec `RANK()` : le premier concentre **22,65 %** des jours d'arrêt prescrits.